# RAG Query: Before and After

This notebook demonstrates the value of RAG by querying an LLM:

1. **Without RAG** -- the LLM answers from its training data alone (often wrong or vague)
2. **With RAG** -- the LLM receives relevant document chunks from Milvus as context (accurate, sourced)

## Prerequisites

- Milvus running with a populated `rag_documents` collection (run `rag_ingestion.ipynb` first)
- A vLLM inference endpoint serving a model (e.g. Mistral 7B Instruct)
- `pymilvus`, `sentence-transformers`, `openai`, and `logfire` installed locally
- Optional: set `LOGFIRE_TOKEN` in your environment for Logfire traces (query notebook)


---
## Step 1 -- Install Dependencies


In [ ]:
%pip install -q pymilvus sentence-transformers openai logfire

---
## Step 2 -- Configure

Set the Milvus and LLM connection details

In [ ]:
# ---------------------------------------------------------------------------
# Milvus
# ---------------------------------------------------------------------------
MILVUS_HOST = "milvus-milvus.milvus.svc.cluster.local"
MILVUS_PORT = 19530
MILVUS_DB = "default"
COLLECTION_NAME = "rag_documents"

# ---------------------------------------------------------------------------
# Embedding (must match the model used during ingestion)
# ---------------------------------------------------------------------------
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# ---------------------------------------------------------------------------
# LLM inference endpoint (vLLM served via KServe)
# ---------------------------------------------------------------------------
INFERENCE_URL = "http://mistral-7b-instruct-metrics.rag-example.svc.cluster.local:8080/v1"
MODEL_NAME = "mistral-7b-instruct"

# ---------------------------------------------------------------------------
# RAG settings
# ---------------------------------------------------------------------------
TOP_K = 5  # Number of chunks to retrieve

print(f"Milvus:     {MILVUS_HOST}:{MILVUS_PORT}/{MILVUS_DB}.{COLLECTION_NAME}")
print(f"Embedding:  {EMBEDDING_MODEL}")
print(f"LLM:        {INFERENCE_URL} ({MODEL_NAME})")
print(f"Top-K:      {TOP_K}")

---
## Step 3 -- Initialize Clients

Connect to Milvus, load the embedding model, and set up the LLM client.


In [ ]:
import logfire
from openai import OpenAI
from pymilvus import MilvusClient
from sentence_transformers import SentenceTransformer

logfire.configure(service_name="rag-query")

# Milvus client
milvus = MilvusClient(uri=f"http://{MILVUS_HOST}:{MILVUS_PORT}", db_name=MILVUS_DB)

# Verify collection exists
if milvus.has_collection(COLLECTION_NAME):
    milvus.load_collection(COLLECTION_NAME)
    stats = milvus.get_collection_stats(COLLECTION_NAME)
else:
    raise RuntimeError(f"Collection '{COLLECTION_NAME}' not found. Run rag_ingestion.ipynb first.")

# Embedding model (same one used during ingestion)
embed_model = SentenceTransformer(EMBEDDING_MODEL)

# LLM client (OpenAI-compatible API)
llm = OpenAI(base_url=INFERENCE_URL, api_key="unused")

---
## Step 4 -- Define Helper Functions


In [ ]:
def ask_llm(question: str, context: str = "") -> str:
    """Send a question to the LLM, optionally with RAG context."""
    mode = "with-rag" if context else "without-rag"
    with logfire.span("llm-query {mode}", mode=mode, model=MODEL_NAME) as span:
        if context:
            prompt = (
                "You are a helpful assistant. Answer the user's question based on the "
                "provided context documents. If the answer is not in the context, say so.\n\n"
                f"## Context\n\n{context}\n\n"
                f"## Question\n\n{question}\n\n"
                "## Answer\n\n"
            )
        else:
            prompt = (
                "You are a helpful assistant. You do NOT have access to external documents "
                "or research papers. If the user asks about specific research findings, "
                "you MUST say: 'I do not have access to research documents. Please provide "
                "context from the relevant papers.'\n\n"
                f"## Question\n\n{question}\n\n"
                "## Answer\n\n"
            )

        response = llm.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=1024,
            temperature=0.1,
        )
        answer = response.choices[0].message.content
        span.set_attribute("prompt_tokens", response.usage.prompt_tokens)
        span.set_attribute("completion_tokens", response.usage.completion_tokens)
        span.set_attribute("answer_length", len(answer))
    return answer


def search_milvus(question: str, top_k: int = TOP_K) -> list:
    """Embed the question and search Milvus for similar chunks."""
    with logfire.span("milvus-search", question=question, top_k=top_k,
                      collection=COLLECTION_NAME) as span:
        query_embedding = embed_model.encode([question], normalize_embeddings=True).tolist()

        results = milvus.search(
            collection_name=COLLECTION_NAME,
            data=query_embedding,
            limit=top_k,
            output_fields=["source_file", "chunk_index", "text"],
            search_params={"metric_type": "COSINE", "params": {"nprobe": 16}},
        )

        contexts = []
        for hits in results:
            for hit in hits:
                contexts.append({
                    "text": hit["entity"]["text"],
                    "source_file": hit["entity"]["source_file"],
                    "chunk_index": hit["entity"]["chunk_index"],
                    "score": hit["distance"],
                })

        span.set_attribute("results_count", len(contexts))
        if contexts:
            span.set_attribute("top_score", round(contexts[0]["score"], 3))
            span.set_attribute("top_source", contexts[0]["source_file"])
    return contexts


def build_context(chunks: list) -> str:
    """Format retrieved chunks into a context string for the prompt."""
    return "\n\n---\n\n".join(
        f"[Source: {c['source_file']}, chunk {c['chunk_index']}, "
        f"score: {c['score']:.3f}]\n{c['text']}"
        for c in chunks
    )


print("Helper functions ready.")

---
## Step 5 -- Pick a Question

Choose a question that the LLM is unlikely to answer correctly from its
training data alone -- something specific to the documents you ingested.


In [44]:
QUESTION = "What is the estimated bifurcation temperature for the circadian oscillator based on the control parameter?"

print(f"Question: {QUESTION}")

Question: What is the estimated bifurcation temperature for the circadian oscillator based on the control parameter?


---
## Step 6 -- Query Without RAG

Ask the LLM directly, with no retrieved context. The model must rely
entirely on its training data.


In [45]:
print(f"Question: {QUESTION}")
print("\n" + "=" * 60)
print("ANSWER (without RAG)")
print("=" * 60)

answer_no_rag = ask_llm(QUESTION)
print(answer_no_rag)

Question: What is the estimated bifurcation temperature for the circadian oscillator based on the control parameter?

ANSWER (without RAG)
12:04:47.226 llm-query without-rag
 I do not have access to research documents. Please provide context from the relevant papers to find the estimated bifurcation temperature for the circadian oscillator based on the control parameter.


---
## Step 7 -- Retrieve Context from Milvus

Embed the question using the same model that was used during ingestion,
then search Milvus for the most similar document chunks.


In [46]:
chunks = search_milvus(QUESTION)

print(f"Retrieved {len(chunks)} chunks from Milvus:\n")
for i, c in enumerate(chunks, 1):
    print(f"  {i}. {c['source_file']} (chunk {c['chunk_index']}, score: {c['score']:.3f})")
    print(f"     {c['text'][:120]}...\n")

12:04:58.163 milvus-search
Retrieved 5 chunks from Milvus:

  1. 2409.05537v1.pdf (chunk 56, score: 0.807)
     and fitted each individual transitory track. The fit shown in Fig. S3, and the extracted parameters 𝜔𝑐 𝑖𝑛 𝑣𝑖𝑡𝑟𝑜 (𝑇) and ...

  2. 2409.05537v1.pdf (chunk 54, score: 0.800)
     We performed the same analysis for all the tracks in our experiments and Figures 3 a ,b show the temperature dependence ...

  3. 2409.05537v1.pdf (chunk 3, score: 0.695)
     temperature  step-down)  allows  for  extracting  and comparing both clock's responses to a temperature step down. It ap...

  4. 2409.05537v1.pdf (chunk 36, score: 0.654)
     directly relates to the dynamical state of the in vivo circadian oscillator 18, 19 . Figures 2 f, g, and h show an examp...

  5. 2409.05537v1.pdf (chunk 77, score: 0.649)
     By  tracking  only  "alive"  cells,  we  found  that  the in  vivo circadian  oscillator  undergoes  a supercritical Hop...



---
## Step 8 -- Query With RAG

Ask the same question, but this time include the retrieved document
chunks as context in the prompt.


In [47]:
context = build_context(chunks)

print(f"Question: {QUESTION}")
print("\n" + "=" * 60)
print("ANSWER (with RAG)")
print("=" * 60)

answer_with_rag = ask_llm(QUESTION, context=context)
print(answer_with_rag)

Question: What is the estimated bifurcation temperature for the circadian oscillator based on the control parameter?

ANSWER (with RAG)
12:05:09.166 llm-query with-rag
 The estimated bifurcation temperature for the circadian oscillator is 21.4°C ± 0.6°C (at 95% confidence interval) based on the control parameter.


---
## Step 9 -- Compare Side by Side


In [48]:
print(f"Question: {QUESTION}")
print("\n" + "=" * 60)
print("WITHOUT RAG")
print("=" * 60)
print(answer_no_rag)

print("\n" + "=" * 60)
print("WITH RAG")
print("=" * 60)
print(answer_with_rag)

print("\n" + "=" * 60)
print("SOURCES")
print("=" * 60)
for c in chunks:
    print(f"  - {c['source_file']} (chunk {c['chunk_index']}, score: {c['score']:.3f})")

Question: What is the estimated bifurcation temperature for the circadian oscillator based on the control parameter?

WITHOUT RAG
 I do not have access to research documents. Please provide context from the relevant papers to find the estimated bifurcation temperature for the circadian oscillator based on the control parameter.

WITH RAG
 The estimated bifurcation temperature for the circadian oscillator is 21.4°C ± 0.6°C (at 95% confidence interval) based on the control parameter.

SOURCES
  - 2409.05537v1.pdf (chunk 56, score: 0.807)
  - 2409.05537v1.pdf (chunk 54, score: 0.800)
  - 2409.05537v1.pdf (chunk 3, score: 0.695)
  - 2409.05537v1.pdf (chunk 36, score: 0.654)
  - 2409.05537v1.pdf (chunk 77, score: 0.649)
